# Run Cosmos T1 GGUF (Windows)

Execution order:
1. Load the model from `C:\Users\Desktop\llm_models\gguf_cache`
2. Chat in the notebook with widgets


In [1]:
import ctypes
import os
from pathlib import Path

os.add_dll_directory(r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.4\bin")
llama_lib_dir = Path(r"C:\Users\Desktop\Desktop\llm_server_venv\Lib\site-packages\llama_cpp\lib")
os.add_dll_directory(str(llama_lib_dir))
ctypes.CDLL(str(llama_lib_dir / "ggml-base.dll"))
ctypes.CDLL(str(llama_lib_dir / "ggml-cpu.dll"))
ctypes.CDLL(str(llama_lib_dir / "ggml-cuda.dll"))
ctypes.CDLL(str(llama_lib_dir / "ggml.dll"))
ctypes.CDLL(str(llama_lib_dir / "llama.dll"))

from run_cosmos_t1_gguf import (
    CONFIG,
    build_prompt,
    create_model,
    load_history,
    sanitize_assistant_text,
    save_history,
)

project_root = Path.cwd()
save_load_path = Path(r"C:\Users\Desktop\llm_models\gguf_cache")

model = create_model(str(save_load_path), use_gpu=True, gpu_layers=-1)
history = load_history()

print(f"Project root: {project_root}")
print(f"Model cache: {save_load_path}")


Backend mode: GPU offload enabled (n_gpu_layers=-1)
Loading cached GGUF: C:\Users\Desktop\llm_models\gguf_cache\Turkish-Gemma-9b-T1.Q4_K_M.gguf


llama_context: n_ctx_per_seq (4096) < n_ctx_train (8192) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


Project root: g:\Drive'ım\training-embedding
Model cache: C:\Users\Desktop\llm_models\gguf_cache


In [2]:
from llama_cpp import Llama
v = Llama(
    model_path=r"C:\Users\Desktop\llm_models\gguf_cache\Turkish-Gemma-9b-T1.Q4_K_M.gguf",
    n_ctx=4096,
    n_threads=4,
    n_gpu_layers=20,
    verbose=True,
)


llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GeForce RTX 3060) - 3905 MiB free
llama_model_loader: loaded meta data with 39 key-value pairs and 464 tensors from C:\Users\Desktop\llm_models\gguf_cache\Turkish-Gemma-9b-T1.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Model
llama_model_loader: - kv   3:                         general.size_label str              = 9.2B
llama_model_loader: - kv   4:                            general.license str              = gemma
llama_model_loader: - kv   5:                   general.base_model.count u32              = 1
llama_model_loader: - kv   6:               

In [3]:
import ipywidgets as widgets
from IPython.display import display

input_box = widgets.Text(
    placeholder="Type your message...",
    description="You:",
    layout=widgets.Layout(width="100%"),
)
send_button = widgets.Button(description="Send")
reset_button = widgets.Button(description="Reset")
output = widgets.Output()


def submit(_):
    global history
    user_input = input_box.value.strip()
    if not user_input:
        return

    input_box.value = ""
    prompt = build_prompt(history, user_input)
    response = model(
        prompt,
        max_tokens=CONFIG["max_tokens"],
        temperature=CONFIG["temperature"],
        top_p=CONFIG["top_p"],
        top_k=CONFIG["top_k"],
        min_p=CONFIG["min_p"],
        repeat_penalty=CONFIG["repeat_penalty"],
        stop=["<end_of_turn>"],
    )
    assistant_text = sanitize_assistant_text(response["choices"][0]["text"].strip())

    history.append((user_input, assistant_text))
    history = history[-CONFIG["max_history_turns"]:]
    save_history(history)

    with output:
        print(f"You: {user_input}")
        print(f"Assistant: {assistant_text}\n")


def reset(_):
    global history
    history = []
    save_history(history)
    output.clear_output()


send_button.on_click(submit)
reset_button.on_click(reset)
input_box.on_submit(submit)

display(input_box, widgets.HBox([send_button, reset_button]), output)


C:\Users\Desktop\AppData\Local\Temp\ipykernel_1856\2948337047.py:52: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_box.on_submit(submit)


Text(value='', description='You:', layout=Layout(width='100%'), placeholder='Type your message...')

Output()